# 03 — AI Career Mentor (RAG) Prototype

1. Load + chunk the career notes with `src.parsing.loader.load_folder('data/career_notes')`.
2. Embed the chunks and build a **FAISS** index; save it to `config.NOTES_INDEX_DIR`.
3. Build the RAG chain with **LangChain**: retrieve top-K chunks for a question, stuff them into the mentor prompt, generate an answer that stays inside the notes.
4. Test three questions: one answerable from the notes, one not (it must refuse / say "I don't know"), and one that needs two notes together.
5. Add the guardrail check (`src/safety/guardrails.py`) before the chain. Move the working chain into `src/mentor/rag_chain.py`.

In [1]:
# ============================================================
# SMART HIRE - NOTEBOOK 03
# AI CAREER MENTOR - RAG PROTOTYPE
#
# Career Notes
#      ↓
# Load Documents
#      ↓
# Chunk Documents
#      ↓
# Local BGE Embeddings (384 dimensions)
#      ↓
# FAISS Index
#      ↓
# Candidate Question
#      ↓
# Guardrail Check
#      ↓
# Retrieve Top-K Notes
#      ↓
# LangChain + Gemini
#      ↓
# Answer ONLY from Career Notes
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import sys
import os
import json
from pathlib import Path

import numpy as np
import faiss

from dotenv import load_dotenv

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate


# ============================================================
# 2. FIND SMART HIRE PROJECT ROOT
# ============================================================

current = Path.cwd()

for folder in [current] + list(current.parents):

    if (
        (folder / "src").is_dir()
        and (folder / "data").is_dir()
        and (folder / "notebooks").is_dir()
    ):
        PROJECT_ROOT = folder
        break

else:
    raise FileNotFoundError(
        "Could not find Smart Hire project root. "
        "Make sure this notebook is inside the Smart_hire_Gen_AI_June project."
    )


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


print("=" * 70)
print("SMART HIRE PROJECT")
print("=" * 70)
print("Project root:", PROJECT_ROOT)


# ============================================================
# 3. LOAD ENVIRONMENT VARIABLES
# ============================================================


env_file = PROJECT_ROOT / ".env.example"

if env_file.exists():
    load_dotenv(
        env_file,
        override=True
    )

else:
    raise FileNotFoundError(
        f".env.example file was not found at:\n{env_file}"
    )


# ============================================================
# 4. IMPORT PROJECT MODULES
# ============================================================

from src import config

from src.parsing.loader import load_folder

from src.search.embed import (
    embed_text,
    embed_texts
)

from src.safety.guardrails import (
    check_question,
    check_answer
)

from src.generate.prompts import MENTOR_SYSTEM_PROMPT


# ============================================================
# 5. DISPLAY CONFIGURATION
# ============================================================

print("\n" + "=" * 70)
print("CONFIGURATION")
print("=" * 70)

print("Career notes folder :", config.CAREER_NOTES_DIR)
print("Notes index folder  :", config.NOTES_INDEX_DIR)
print("Embedding model     :", config.EMBED_MODEL)
print("Embedding dimension :", config.EMBED_DIM)
print("Chat model          :", config.CHAT_MODEL)
print("Chunk size          :", config.CHUNK_SIZE)
print("Chunk overlap       :", config.CHUNK_OVERLAP)
print("Top-K notes         :", config.TOP_K_NOTES)


# ============================================================
# 6. VERIFY GEMINI API KEY
# ============================================================

api_key = os.getenv(config.API_KEY_ENV)

if not api_key:
    raise ValueError(
        f"{config.API_KEY_ENV} was not found.\n"
        "Please make sure your Gemini API key is present in .env.example."
    )

print("\nGemini API key found.")


# ============================================================
# 7. VERIFY CAREER NOTES FOLDER
# ============================================================

career_notes_folder = config.CAREER_NOTES_DIR

if not career_notes_folder.exists():
    raise FileNotFoundError(
        f"Career notes folder not found:\n"
        f"{career_notes_folder}"
    )


note_files = [
    file
    for file in career_notes_folder.iterdir()
    if file.is_file()
]

if not note_files:
    raise ValueError(
        f"No career note files found in:\n"
        f"{career_notes_folder}"
    )


print("\n" + "=" * 70)
print("CAREER NOTES")
print("=" * 70)

print("Folder:", career_notes_folder)
print("Files found:", len(note_files))

for file in note_files:
    print(" -", file.name)


# ============================================================
# 8. LOAD CAREER NOTES
# ============================================================

print("\nLoading career notes...")

documents = load_folder(
    career_notes_folder
)

if not documents:
    raise ValueError(
        "load_folder() did not return any documents."
    )


print("Documents loaded:", len(documents))


# ============================================================
# 9. CHUNK CAREER NOTES
# ============================================================

print("\nCreating chunks...")

from langchain_core.documents import Document

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=config.CHUNK_SIZE,
    chunk_overlap=config.CHUNK_OVERLAP
)

# Convert tuples returned by load_folder()
# into LangChain Document objects
langchain_documents = []

for item in documents:

    if isinstance(item, tuple):

        text, metadata = item

        langchain_documents.append(
            Document(
                page_content=str(text),
                metadata=metadata if isinstance(metadata, dict) else {}
            )
        )

    else:

        langchain_documents.append(item)


chunks = text_splitter.split_documents(
    langchain_documents
)

if not chunks:
    raise ValueError(
        "No chunks were created from the career notes."
    )

print("Original documents:", len(documents))
print("LangChain documents:", len(langchain_documents))
print("Total chunks:", len(chunks))


# ============================================================
# 10. PREPARE CHUNK TEXTS
# ============================================================

chunk_texts = [
    chunk.page_content.strip()
    for chunk in chunks
    if chunk.page_content
    and chunk.page_content.strip()
]

if not chunk_texts:
    raise ValueError(
        "All career-note chunks are empty."
    )


print("Usable chunk texts:", len(chunk_texts))


# ============================================================
# 11. TEST LOCAL EMBEDDING
# ============================================================

print("\nTesting local embedding model...")

test_embedding = embed_text(
    chunk_texts[0]
)

test_embedding = np.asarray(
    test_embedding,
    dtype=np.float32
)

print("Test embedding shape:", test_embedding.shape)

if test_embedding.shape != (
    config.EMBED_DIM,
):
    raise ValueError(
        f"Embedding dimension mismatch.\n"
        f"Expected: {config.EMBED_DIM}\n"
        f"Received: {test_embedding.shape}"
    )


print("Embedding model verified successfully.")


# ============================================================
# 12. CREATE EMBEDDINGS FOR ALL CHUNKS
# ============================================================

print("\n" + "=" * 70)
print("CREATING NOTE EMBEDDINGS")
print("=" * 70)

note_embeddings = embed_texts(
    chunk_texts
)

note_embeddings = np.asarray(
    note_embeddings,
    dtype=np.float32
)


if note_embeddings.ndim != 2:
    raise ValueError(
        "Embeddings must be a 2-dimensional array."
    )


if note_embeddings.shape[0] != len(chunk_texts):
    raise ValueError(
        "Number of embeddings does not match "
        "number of chunks."
    )


if note_embeddings.shape[1] != config.EMBED_DIM:
    raise ValueError(
        f"Expected {config.EMBED_DIM}-dimensional embeddings, "
        f"but received {note_embeddings.shape[1]}."
    )


print("Number of chunks:", len(chunk_texts))
print("Embedding shape:", note_embeddings.shape)
print("Embedding dimension:", note_embeddings.shape[1])


# ============================================================
# 13. CREATE FAISS INDEX
# ============================================================

print("\n" + "=" * 70)
print("CREATING FAISS INDEX")
print("=" * 70)

embedding_dimension = note_embeddings.shape[1]

# Embeddings are normalized in embed.py.
# Therefore Inner Product is equivalent to cosine similarity.
notes_index = faiss.IndexFlatIP(
    embedding_dimension
)

notes_index.add(
    note_embeddings
)


print("FAISS index created.")
print("Vectors stored:", notes_index.ntotal)
print("FAISS dimension:", notes_index.d)


# ============================================================
# 14. VERIFY FAISS INDEX
# ============================================================

if notes_index.ntotal != len(chunk_texts):
    raise ValueError(
        "FAISS vector count does not match "
        "the number of chunks."
    )


if notes_index.d != config.EMBED_DIM:
    raise ValueError(
        f"FAISS dimension mismatch.\n"
        f"Expected: {config.EMBED_DIM}\n"
        f"Received: {notes_index.d}"
    )


print("FAISS index verification successful.")


# ============================================================
# 15. CREATE NOTES INDEX DIRECTORY
# ============================================================

config.NOTES_INDEX_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 16. SAVE FAISS INDEX
# ============================================================

notes_index_file = (
    config.NOTES_INDEX_DIR / "notes.faiss"
)

faiss.write_index(
    notes_index,
    str(notes_index_file)
)


print("\nFAISS index saved to:")
print(notes_index_file)


# ============================================================
# 17. SAVE CHUNK TEXTS + METADATA
# ============================================================

notes_metadata_file = (
    config.NOTES_INDEX_DIR / "notes.json"
)

notes_metadata = []

for chunk in chunks:

    text = chunk.page_content.strip()

    if not text:
        continue

    notes_metadata.append({
        "text": text,
        "metadata": chunk.metadata
    })


with open(
    notes_metadata_file,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        notes_metadata,
        file,
        ensure_ascii=False,
        indent=2
    )


print("Notes metadata saved to:")
print(notes_metadata_file)


# ============================================================
# 18. LOAD FAISS INDEX FROM DISK
# 
# This verifies that the application can load the saved
# index instead of rebuilding it every time.
# ============================================================

print("\n" + "=" * 70)
print("TESTING SAVED FAISS INDEX")
print("=" * 70)

loaded_notes_index = faiss.read_index(
    str(notes_index_file)
)


with open(
    notes_metadata_file,
    "r",
    encoding="utf-8"
) as file:

    loaded_metadata = json.load(file)


loaded_note_texts = [
    item["text"]
    for item in loaded_metadata
]


if loaded_notes_index.ntotal != len(
    loaded_note_texts
):
    raise ValueError(
        "Loaded FAISS index and metadata count do not match."
    )


if loaded_notes_index.d != config.EMBED_DIM:
    raise ValueError(
        "Loaded FAISS index has the wrong embedding dimension."
    )


print("Saved FAISS index loaded successfully.")
print("Vectors:", loaded_notes_index.ntotal)
print("Dimension:", loaded_notes_index.d)
print("Metadata chunks:", len(loaded_note_texts))


# ============================================================
# 19. RETRIEVAL FUNCTION
# ============================================================

def retrieve_notes(
    question: str,
    index,
    note_texts: list[str],
    top_k: int = None
):

    """
    Retrieve the most relevant career-note chunks
    for a user's question.
    """

    if not question or not question.strip():
        return []


    if top_k is None:
        top_k = config.TOP_K_NOTES


    top_k = min(
        top_k,
        index.ntotal
    )


    question_embedding = embed_text(
        question
    )

    question_embedding = np.asarray(
        question_embedding,
        dtype=np.float32
    )


    if question_embedding.shape != (
        config.EMBED_DIM,
    ):
        raise ValueError(
            f"Question embedding has incorrect dimension: "
            f"{question_embedding.shape}"
        )


    question_embedding = question_embedding.reshape(
        1,
        -1
    )


    scores, indices = index.search(
        question_embedding,
        top_k
    )


    retrieved = []


    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        if idx == -1:
            continue


        retrieved.append({
            "text": note_texts[idx],
            "score": float(score),
            "index": int(idx)
        })


    return retrieved


# ============================================================
# 20. LOAD LANGCHAIN GEMINI MODEL
# ============================================================

print("\n" + "=" * 70)
print("LOADING LANGCHAIN GEMINI")
print("=" * 70)

llm = ChatGoogleGenerativeAI(
    model=config.CHAT_MODEL,
    temperature=0
)

print("LangChain Gemini loaded successfully.")


# ============================================================
# 21. CREATE MENTOR PROMPT
# ============================================================

mentor_prompt = ChatPromptTemplate.from_template(
    MENTOR_SYSTEM_PROMPT
)


# ============================================================
# 22. CREATE LANGCHAIN RAG CHAIN
# ============================================================

rag_chain = mentor_prompt | llm

print("LangChain RAG chain created successfully.")


# ============================================================
# 23. AI CAREER MENTOR FUNCTION
# ============================================================

def answer_question(
    question: str,
    index,
    note_texts: list[str],
    top_k: int = None
):

    """
    Complete AI Career Mentor pipeline:

    Question
        ↓
    Guardrail
        ↓
    Retrieval
        ↓
    Context
        ↓
    LangChain + Gemini
        ↓
    Answer guardrail
    """


    # --------------------------------------------------------
    # QUESTION GUARDRAIL
    # --------------------------------------------------------

    allowed, message = check_question(
        question
    )


    if not allowed:

        return {
            "answer": message,
            "sources": []
        }


    # --------------------------------------------------------
    # RETRIEVE RELEVANT NOTES
    # --------------------------------------------------------

    retrieved = retrieve_notes(
        question=question,
        index=index,
        note_texts=note_texts,
        top_k=top_k
    )


    if not retrieved:

        return {
            "answer": (
                "I don't know based on the "
                "provided career notes."
            ),
            "sources": []
        }


    # --------------------------------------------------------
    # BUILD CONTEXT
    # --------------------------------------------------------

    context_parts = []


    for number, item in enumerate(
        retrieved,
        start=1
    ):

        context_parts.append(
            f"[Note {number}]\n"
            f"{item['text']}"
        )


    context = "\n\n".join(
        context_parts
    )


    # --------------------------------------------------------
    # RUN LANGCHAIN CHAIN
    # --------------------------------------------------------

    response = rag_chain.invoke({
        "context": context,
        "question": question
    })


    # --------------------------------------------------------
    # EXTRACT ANSWER
    # --------------------------------------------------------

    if hasattr(
        response,
        "content"
    ):

        answer = response.content

    else:

        answer = str(response)


    # Handle possible non-string response content
    if not isinstance(
        answer,
        str
    ):

        answer = str(answer)


    answer = answer.strip()


    # --------------------------------------------------------
    # ANSWER GUARDRAIL
    # --------------------------------------------------------

    answer = check_answer(
        answer
    )


    return {
        "answer": answer,
        "sources": retrieved
    }


# ============================================================
# 24. HELPER FUNCTION TO DISPLAY RESULTS
# ============================================================

def display_result(
    title,
    question,
    result
):

    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

    print("\nQuestion:")
    print(question)

    print("\nAnswer:")
    print(result["answer"])

    print("\nRetrieved sources:")

    if not result["sources"]:

        print("No sources retrieved.")

        return


    for number, source in enumerate(
        result["sources"],
        start=1
    ):

        print(
            f"\n--- Source {number} "
            f"(score: {source['score']:.4f}) ---"
        )

        print(
            source["text"][:700]
        )


# ============================================================
# 25. TEST 1 - ANSWERABLE QUESTION
#
# IMPORTANT:
# Replace this question with a question whose answer is
# definitely present in your career notes.
# ============================================================

question_1 = """
What skills are recommended for starting a career in data science?
"""


result_1 = answer_question(
    question=question_1,
    index=loaded_notes_index,
    note_texts=loaded_note_texts,
    top_k=config.TOP_K_NOTES
)


display_result(
    "TEST 1 - ANSWERABLE QUESTION",
    question_1,
    result_1
)


# ============================================================
# 26. TEST 2 - UNANSWERABLE QUESTION
#
# This question is intentionally outside the career notes.
# The mentor should say:
#
# "I don't know based on the provided career notes."
# ============================================================

question_2 = """
What will be the exact salary of a data scientist
in India in the year 2030?
"""


result_2 = answer_question(
    question=question_2,
    index=loaded_notes_index,
    note_texts=loaded_note_texts,
    top_k=config.TOP_K_NOTES
)


display_result(
    "TEST 2 - UNANSWERABLE QUESTION",
    question_2,
    result_2
)


# ============================================================
# 27. TEST 3 - MULTI-NOTE QUESTION
#
# IMPORTANT:
# Modify this question so that it requires information from
# TWO different career notes in your data/career_notes folder.
# ============================================================

question_3 = """
How can the skills discussed in different career notes
be combined to prepare for an AI-related career?
"""


result_3 = answer_question(
    question=question_3,
    index=loaded_notes_index,
    note_texts=loaded_note_texts,
    top_k=config.TOP_K_NOTES
)


display_result(
    "TEST 3 - MULTI-NOTE QUESTION",
    question_3,
    result_3
)


# ============================================================
# 28. FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 70)
print("AI CAREER MENTOR - RAG PROTOTYPE COMPLETED")
print("=" * 70)

print("\nOriginal documents :", len(documents))
print("Chunks created     :", len(chunks))
print("Embeddings         :", len(note_embeddings))
print("Embedding dimension:", config.EMBED_DIM)
print("FAISS vectors      :", loaded_notes_index.ntotal)
print("FAISS dimension    :", loaded_notes_index.d)
print("Top-K retrieval    :", config.TOP_K_NOTES)

print("\nEmbedding model:")
print(config.EMBED_MODEL)

print("\nChat model:")
print(config.CHAT_MODEL)

print("\nFAISS index:")
print(notes_index_file)

print("\nNotes metadata:")
print(notes_metadata_file)

print("\nRAG components:")
print("✓ Career notes loaded")
print("✓ Notes chunked")
print("✓ Local embeddings created")
print("✓ FAISS index created")
print("✓ FAISS index saved")
print("✓ FAISS index loaded")
print("✓ Guardrail check added")
print("✓ LangChain Gemini chain created")
print("✓ Answerable question tested")
print("✓ Unanswerable question tested")
print("✓ Multi-note question tested")

print("\n" + "=" * 70)
print("TASK #03 PROTOTYPE READY")
print("=" * 70)

c:\Users\jeesh\Smart_hire_Gen_AI_June\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SMART HIRE PROJECT
Project root: c:\Users\jeesh\Smart_hire_Gen_AI_June


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5616.98it/s]



CONFIGURATION
Career notes folder : C:\Users\jeesh\Smart_hire_Gen_AI_June\data\career_notes
Notes index folder  : C:\Users\jeesh\Smart_hire_Gen_AI_June\vectorstore\notes_faiss
Embedding model     : BAAI/bge-small-en-v1.5
Embedding dimension : 384
Chat model          : gemini-3.5-flash-lite
Chunk size          : 800
Chunk overlap       : 150
Top-K notes         : 3

Gemini API key found.

CAREER NOTES
Folder: C:\Users\jeesh\Smart_hire_Gen_AI_June\data\career_notes
Files found: 2
 - data_analyst_roadmap.md
 - resume_writing_tips.md

Loading career notes...
Documents loaded: 5

Creating chunks...
Original documents: 5
LangChain documents: 5
Total chunks: 5
Usable chunk texts: 5

Testing local embedding model...
Test embedding shape: (384,)
Embedding model verified successfully.

CREATING NOTE EMBEDDINGS


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.05it/s]


...

LangChain Gemini loaded successfully.
LangChain RAG chain created successfully.

TEST 1 - ANSWERABLE QUESTION

Question:

What skills are recommended for starting a career in data science?


Answer:
[{'type': 'text', 'text': "I don't know based on the provided career notes.", 'extras': {'signature': 'El4KXAERTTIP2JvtBj6uco0TNaOMBRpy96fre6Bto/9o1V/A98kkCKfY9dpjCfWqdo3u3VhXuv9nlekC9h7eyWYltPd0FC9B5a2DWuD730bP8ZPi+El+/pt9JatS84Dj'}}]

Retrieved sources:

--- Source 1 (score: 0.7867) ---
# How to Become a Data Analyst A data analyst collects, cleans, and interprets data to help a business make decisions. It is one of the most common first roles for people moving into data. ## Core skills - **SQL** — the single most important skill. You must be able to write SELECT queries with JOINs, GROUP BY, and filtering. - **Spreadsheets** — Excel or Google Sheets: pivot tables, lookups, charts. - **A visualisation tool** — Power BI or Tableau to build dashboards. - **Statistics basics** — averages, di

c:\Users\jeesh\Smart_hire_Gen_AI_June\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



TEST 2 - UNANSWERABLE QUESTION

Question:

What will be the exact salary of a data scientist
in India in the year 2030?


Answer:
[{'type': 'text', 'text': "I don't know based on the provided career notes.", 'extras': {'signature': 'El4KXAERTTIP99KZb1maXjNit8WLzJ7KSYthNnn0YQj/5biwJZpSoMSb5vWRIoSkakk4bBH/NofBRnFToXxJuJjeLfgaV6GtRQQQO7XrvYaHv9SzJE472kjEgdUQaMcK'}}]

Retrieved sources:

--- Source 1 (score: 0.6057) ---
lots. ## Switching from another role If you are coming from a non-data job, the fastest path is: 1. Learn SQL first and practise on real datasets. 2. Rebuild reports you already make by hand into a dashboard. 3. Do two or three portfolio projects analysing public datasets. 4. Apply to "junior data analyst" or "business analyst" roles. ## What a data analyst does NOT need at the start You do not need machine learning, deep learning, or a statistics degree to get a first data-analyst job. Those matter more for data-scientist roles later. ## Typical entry-level expectations -

c:\Users\jeesh\Smart_hire_Gen_AI_June\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



TEST 3 - MULTI-NOTE QUESTION

Question:

How can the skills discussed in different career notes
be combined to prepare for an AI-related career?


Answer:
[{'type': 'text', 'text': "I don't know based on the provided career notes.", 'extras': {'signature': 'El4KXAERTTIPNHIWKMFA35cqlMPc3awOSM2ekATKLYOTWU9GNGNwTQWyQtvxuAuQVtOotChqx9mm09qkZn8TWFuj/AEND6qmN3H3m/RpkXE74tSGOmjjhZK6dv1aju/O'}}]

Retrieved sources:

--- Source 1 (score: 0.6768) ---
List the specific tools and languages the job asks for, only if you actually know them. - Group them (Languages, Tools, Databases) so a reader finds them fast. ## Common mistakes - A long paragraph objective instead of a short summary. - Listing every course or skill instead of the relevant ones. - Spelling and formatting errors — proofread, and keep fonts and spacing consistent. ## Tailoring for a specific job Read the job description, note the skills it repeats, and make sure those exact words appear on your resume if they are true for you. This 

In [2]:
# ============================================================
# SMART HIRE - CAREER NOTES FAISS VECTORSTORE
# ============================================================

from pathlib import Path
import sys
import json
import re

import numpy as np
import faiss


# ============================================================
# 1. FIND PROJECT ROOT
# ============================================================

current = Path.cwd()

PROJECT_ROOT = None

for folder in [current] + list(current.parents):
    if (
        (folder / "src").is_dir()
        and (folder / "data").is_dir()
        and (folder / "notebooks").is_dir()
    ):
        PROJECT_ROOT = folder
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the SmartHire project root. "
        "Make sure this notebook is inside the SmartHire project."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ============================================================
# 2. IMPORT PROJECT MODULES
# ============================================================

from src import config
from src.search.embed import embed_texts


# ============================================================
# 3. DEFINE DIRECTORIES
# ============================================================

NOTES_DIR = config.CAREER_NOTES_DIR
NOTES_INDEX_DIR = config.NOTES_INDEX_DIR

NOTES_INDEX_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 70)
print("SMART HIRE - CAREER NOTES VECTORSTORE")
print("=" * 70)

print("Project root:")
print(PROJECT_ROOT)

print()

print("Career notes directory:")
print(NOTES_DIR)

print()

print("Notes FAISS directory:")
print(NOTES_INDEX_DIR)

print()

print("Embedding model:")
print(config.EMBED_MODEL)

print()

print("Embedding dimension:")
print(config.EMBED_DIM)


# ============================================================
# 4. CHECK CAREER NOTES DIRECTORY
# ============================================================

if not NOTES_DIR.exists():
    raise FileNotFoundError(
        f"Career notes directory does not exist:\n{NOTES_DIR}"
    )


# ============================================================
# 5. FIND ALL FILES RECURSIVELY
# ============================================================

files = [
    path
    for path in NOTES_DIR.rglob("*")
    if path.is_file()
]

if len(files) == 0:
    raise FileNotFoundError(
        f"No files were found inside:\n{NOTES_DIR}"
    )


print()
print("=" * 70)
print("FILES FOUND")
print("=" * 70)

print("Total files:", len(files))

for path in files:
    print("-", path.relative_to(NOTES_DIR))


# ============================================================
# 6. SHOW FILE TYPES
# ============================================================

from collections import Counter

extensions = Counter(
    path.suffix.lower() if path.suffix else "[no extension]"
    for path in files
)

print()
print("=" * 70)
print("FILE TYPES")
print("=" * 70)

for extension, count in extensions.items():
    print(f"{extension}: {count}")


# ============================================================
# 7. TEXT CLEANING FUNCTION
# ============================================================

def clean_text(text):
    if text is None:
        return ""

    text = str(text)

    text = text.replace("\x00", " ")

    text = re.sub(
        r"\r\n?",
        "\n",
        text
    )

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()


# ============================================================
# 8. FILE READING FUNCTIONS
# ============================================================

def read_text_file(path):
    return path.read_text(
        encoding="utf-8",
        errors="ignore"
    )


def read_pdf_file(path):
    from pypdf import PdfReader

    reader = PdfReader(str(path))

    pages = []

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            pages.append(page_text)

    return "\n\n".join(pages)


def read_docx_file(path):
    from docx import Document

    document = Document(str(path))

    paragraphs = []

    for paragraph in document.paragraphs:
        text = paragraph.text.strip()

        if text:
            paragraphs.append(text)

    return "\n".join(paragraphs)


def read_json_file(path):
    raw_text = path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    data = json.loads(raw_text)

    return json.dumps(
        data,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 9. GENERAL FILE TEXT EXTRACTION
# ============================================================

def extract_text(path):

    extension = path.suffix.lower()

    try:

        if extension in {
            ".txt",
            ".md",
            ".markdown",
            ".csv"
        }:
            text = read_text_file(path)

        elif extension == ".pdf":
            text = read_pdf_file(path)

        elif extension == ".docx":
            text = read_docx_file(path)

        elif extension == ".json":
            text = read_json_file(path)

        else:
            # Try reading unknown/no-extension files as text.
            text = read_text_file(path)

        return clean_text(text)

    except Exception as error:

        print(
            f"WARNING: Could not read "
            f"{path.name}: {error}"
        )

        return ""


# ============================================================
# 10. LOAD ALL CAREER NOTE DOCUMENTS
# ============================================================

documents = []

print()
print("=" * 70)
print("READING CAREER NOTES")
print("=" * 70)

for path in files:

    text = extract_text(path)

    if not text:
        print(
            "Skipping empty/unreadable file:",
            path.name
        )
        continue

    relative_source = str(
        path.relative_to(NOTES_DIR)
    )

    documents.append(
        {
            "source": relative_source,
            "file_name": path.name,
            "text": text
        }
    )

    print(
        f"Loaded: {relative_source} "
        f"({len(text)} characters)"
    )


if len(documents) == 0:
    raise ValueError(
        "No readable career-note documents were found."
    )


print()
print("Readable documents:", len(documents))


# ============================================================
# 11. CREATE TEXT CHUNKS
# ============================================================

def chunk_text(
    text,
    chunk_size,
    overlap
):

    text = clean_text(text)

    if not text:
        return []

    if overlap >= chunk_size:
        raise ValueError(
            "CHUNK_OVERLAP must be smaller than CHUNK_SIZE."
        )

    chunks = []

    start = 0
    text_length = len(text)

    while start < text_length:

        end = min(
            start + chunk_size,
            text_length
        )

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= text_length:
            break

        start = end - overlap

    return chunks


# ============================================================
# 12. CREATE CHUNKS FOR ALL DOCUMENTS
# ============================================================

note_chunks = []

for document in documents:

    chunks = chunk_text(
        document["text"],
        config.CHUNK_SIZE,
        config.CHUNK_OVERLAP
    )

    for chunk_index, chunk in enumerate(chunks):

        note_chunks.append(
            {
                "source": document["source"],
                "file_name": document["file_name"],
                "chunk_index": chunk_index,
                "text": chunk
            }
        )


if len(note_chunks) == 0:
    raise ValueError(
        "No text chunks were created from the career notes."
    )


print()
print("=" * 70)
print("CHUNKING COMPLETE")
print("=" * 70)

print("Number of source documents:", len(documents))
print("Number of chunks:", len(note_chunks))
print("Chunk size:", config.CHUNK_SIZE)
print("Chunk overlap:", config.CHUNK_OVERLAP)


# ============================================================
# 13. PREPARE TEXTS FOR EMBEDDING
# ============================================================

note_texts = [
    item["text"]
    for item in note_chunks
]

print()
print("Texts ready for embedding:", len(note_texts))


# ============================================================
# 14. CREATE BGE EMBEDDINGS
# ============================================================

print()
print("=" * 70)
print("CREATING EMBEDDINGS")
print("=" * 70)

note_embeddings = embed_texts(
    note_texts,
    batch_size=32
)

print()
print("Embedding shape:", note_embeddings.shape)


# ============================================================
# 15. VALIDATE EMBEDDINGS
# ============================================================

expected_shape = (
    len(note_chunks),
    config.EMBED_DIM
)

if note_embeddings.shape != expected_shape:

    raise ValueError(
        "Unexpected embedding shape.\n"
        f"Expected: {expected_shape}\n"
        f"Received: {note_embeddings.shape}"
    )


if not np.isfinite(note_embeddings).all():

    raise ValueError(
        "Embeddings contain NaN or infinite values."
    )


note_embeddings = np.asarray(
    note_embeddings,
    dtype=np.float32
)


print()
print("Embedding validation successful.")
print("Dimension:", note_embeddings.shape[1])


# ============================================================
# 16. CREATE FAISS INDEX
# ============================================================

print()
print("=" * 70)
print("CREATING FAISS INDEX")
print("=" * 70)

index = faiss.IndexFlatIP(
    config.EMBED_DIM
)

index.add(
    note_embeddings
)


print("FAISS index created.")
print("Number of vectors:", index.ntotal)
print("Index dimension:", index.d)


# ============================================================
# 17. CREATE METADATA
# ============================================================

notes_metadata = []

for index_number, item in enumerate(note_chunks):

    notes_metadata.append(
        {
            "index": index_number,
            "source": item["source"],
            "file_name": item["file_name"],
            "chunk_index": item["chunk_index"],
            "text": item["text"]
        }
    )


if len(notes_metadata) != index.ntotal:

    raise ValueError(
        "Metadata count does not match FAISS vector count."
    )


print()
print("Metadata records:", len(notes_metadata))


# ============================================================
# 18. SAVE NOTES FAISS INDEX
# ============================================================

notes_faiss_path = (
    NOTES_INDEX_DIR / "notes.faiss"
)

faiss.write_index(
    index,
    str(notes_faiss_path)
)


print()
print("Saved FAISS index:")
print(notes_faiss_path)


# ============================================================
# 19. SAVE NOTES JSON
# ============================================================

notes_json_path = (
    NOTES_INDEX_DIR / "notes.json"
)

with open(
    notes_json_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        notes_metadata,
        file,
        ensure_ascii=False,
        indent=2
    )


print()
print("Saved metadata:")
print(notes_json_path)


# ============================================================
# 20. CHECK FILES
# ============================================================

if not notes_faiss_path.exists():

    raise FileNotFoundError(
        "notes.faiss was not created."
    )


if not notes_json_path.exists():

    raise FileNotFoundError(
        "notes.json was not created."
    )


print()
print("=" * 70)
print("FILES CREATED SUCCESSFULLY")
print("=" * 70)

print(
    "notes.faiss:",
    notes_faiss_path
)

print(
    "notes.json:",
    notes_json_path
)


# ============================================================
# 21. RELOAD FAISS INDEX
# ============================================================

print()
print("=" * 70)
print("RELOADING AND VERIFYING")
print("=" * 70)

loaded_index = faiss.read_index(
    str(notes_faiss_path)
)

with open(
    notes_json_path,
    "r",
    encoding="utf-8"
) as file:

    loaded_notes = json.load(file)


print("Reloaded FAISS vectors:", loaded_index.ntotal)
print("Reloaded metadata:", len(loaded_notes))
print("Reloaded dimension:", loaded_index.d)


# ============================================================
# 22. VALIDATE SAVED INDEX
# ============================================================

if loaded_index.d != config.EMBED_DIM:

    raise ValueError(
        "Saved FAISS index has the wrong dimension.\n"
        f"Expected: {config.EMBED_DIM}\n"
        f"Received: {loaded_index.d}"
    )


if loaded_index.ntotal != len(loaded_notes):

    raise ValueError(
        "FAISS vector count does not match metadata count."
    )


if loaded_index.ntotal == 0:

    raise ValueError(
        "FAISS index is empty."
    )


print()
print("Saved index validation successful.")


# ============================================================
# 23. TEST RETRIEVAL
# ============================================================

print()
print("=" * 70)
print("TESTING CAREER NOTE RETRIEVAL")
print("=" * 70)

test_question = (
    "How can I prepare for a data analyst career?"
)

query_embedding = embed_texts(
    [test_question]
)

query_embedding = np.asarray(
    query_embedding,
    dtype=np.float32
)


if query_embedding.shape != (
    1,
    config.EMBED_DIM
):

    raise ValueError(
        "Query embedding has the wrong dimension."
    )


scores, indices = loaded_index.search(
    query_embedding,
    5
)


print()
print("Test question:")
print(test_question)

print()

print("Top retrieved career notes:")


for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]),
    start=1
):

    if idx < 0:
        continue

    note = loaded_notes[idx]

    print()
    print("-" * 70)
    print("Rank:", rank)
    print("Similarity score:", float(score))
    print("Source:", note["source"])
    print("Chunk:", note["chunk_index"])
    print()
    print(
        note["text"][:800]
    )


# ============================================================
# 24. FINAL STATUS
# ============================================================

print()
print()
print("=" * 70)
print("SMART HIRE NOTES VECTORSTORE READY")
print("=" * 70)

print()
print("Source documents:", len(documents))
print("Total chunks:", len(note_chunks))
print("Embedding dimension:", config.EMBED_DIM)
print("FAISS vectors:", loaded_index.ntotal)

print()
print("Created files:")

print(
    NOTES_INDEX_DIR / "notes.faiss"
)

print(
    NOTES_INDEX_DIR / "notes.json"
)

print()
print("STATUS: SUCCESS")
print("Career Mentor notes vectorstore is ready.")

c:\Users\jeesh\Smart_hire_Gen_AI_June\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1629.79it/s]


SMART HIRE - CAREER NOTES VECTORSTORE
Project root:
c:\Users\jeesh\Smart_hire_Gen_AI_June

Career notes directory:
C:\Users\jeesh\Smart_hire_Gen_AI_June\data\career_notes

Notes FAISS directory:
C:\Users\jeesh\Smart_hire_Gen_AI_June\vectorstore\notes_faiss

Embedding model:
BAAI/bge-small-en-v1.5

Embedding dimension:
384

FILES FOUND
Total files: 12
- data_analyst_roadmap.md
- resume_writing_tips.md
- carrer_guides\AI_Engineer_Roadmap_2025.md
- carrer_guides\Backend_Developer_Roadmap.md
- carrer_guides\Career_Roadmap_for_Freshers.md
- carrer_guides\Data_Analyst_Role_Guide.md
- carrer_guides\Frontend_Developer_Roadmap.md
- carrer_guides\Full_Stack_Developer_Role_Guide.md
- carrer_guides\GenAI_Developer_Roadmap.md
- carrer_guides\Interview_Preparation_Guide.md
- carrer_guides\ML_Engineer_Roadmap.md
- carrer_guides\Resume_and_Portfolio_Guide.md

FILE TYPES
.md: 12

READING CAREER NOTES
Loaded: data_analyst_roadmap.md (1410 characters)
Loaded: resume_writing_tips.md (1273 characters)
Load

Batches: 100%|██████████| 3/3 [00:10<00:00,  3.60s/it]



Embedding shape: (82, 384)

Embedding validation successful.
Dimension: 384

CREATING FAISS INDEX
FAISS index created.
Number of vectors: 82
Index dimension: 384

Metadata records: 82

Saved FAISS index:
C:\Users\jeesh\Smart_hire_Gen_AI_June\vectorstore\notes_faiss\notes.faiss

Saved metadata:
C:\Users\jeesh\Smart_hire_Gen_AI_June\vectorstore\notes_faiss\notes.json

FILES CREATED SUCCESSFULLY
notes.faiss: C:\Users\jeesh\Smart_hire_Gen_AI_June\vectorstore\notes_faiss\notes.faiss
notes.json: C:\Users\jeesh\Smart_hire_Gen_AI_June\vectorstore\notes_faiss\notes.json

RELOADING AND VERIFYING
Reloaded FAISS vectors: 82
Reloaded metadata: 82
Reloaded dimension: 384

Saved index validation successful.

TESTING CAREER NOTE RETRIEVAL


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.52it/s]


Test question:
How can I prepare for a data analyst career?

Top retrieved career notes:

----------------------------------------------------------------------
Rank: 1
Similarity score: 0.8329559564590454
Source: data_analyst_roadmap.md
Chunk: 0

# How to Become a Data Analyst

A data analyst collects, cleans, and interprets data to help a business make
decisions. It is one of the most common first roles for people moving into data.

## Core skills
- **SQL** — the single most important skill. You must be able to write SELECT
 queries with JOINs, GROUP BY, and filtering.
- **Spreadsheets** — Excel or Google Sheets: pivot tables, lookups, charts.
- **A visualisation tool** — Power BI or Tableau to build dashboards.
- **Statistics basics** — averages, distributions, correlation, and how to
 spot a misleading chart.
- **Python (optional but valued)** — pandas for cleaning and matplotlib for plots.

## Switching from another role
If you are coming from a non-data job, the fastest path is: